## Load the Data

In [1]:
import pandas as pd

raw_data_path = 'raw_data/DataCoSupplyChainDataset.csv'

raw_data = pd.read_csv(
    raw_data_path,
    encoding='latin-1'
)

raw_data = raw_data.sort_values(
    by=['Order Id', 'Order Item Id'],
    ascending=[True, True]
)

print('raw_data in', raw_data_path, 'has been loaded successfully')

raw_data in raw_data/DataCoSupplyChainDataset.csv has been loaded successfully


## Data Preview

In [2]:
raw_data.head(5)

,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
33833,CASH,2,4,88.790001,239.979996,Advance shipping,0,43,Camping & Hiking,Hickory,...,NaN,957,43,NaN,http://images.acmesports.sports/Diamondback+Wo...,Diamondback Women's Serene Classic Comfort Bi,299.980011,0,1/3/2015 0:00,Standard Class
77011,PAYMENT,3,4,91.180000,193.990005,Advance shipping,0,48,Water Sports,Chicago,...,NaN,1073,48,NaN,http://images.acmesports.sports/Pelican+Sunstr...,Pelican Sunstream 100 Kayak,199.990005,0,1/4/2015 0:21,Standard Class
109322,PAYMENT,3,4,68.250000,227.500000,Advance shipping,0,24,Women's Apparel,Chicago,...,NaN,502,24,NaN,http://images.acmesports.sports/Nike+Men%27s+D...,Nike Men's Dri-FIT Victory Golf Polo,50.000000,0,1/4/2015 0:21,Standard Class
87884,PAYMENT,3,4,36.470001,107.889999,Advance shipping,0,18,Men's Footwear,Chicago,...,NaN,403,18,NaN,http://images.acmesports.sports/Nike+Men%27s+C...,Nike Men's CJ Elite 2 TD Football Cleat,129.990005,0,1/4/2015 0:21,Standard Class
63764,CASH,5,4,4.100000,40.980000,Late delivery,1,40,Accessories,San Antonio,...,NaN,897,40,NaN,http://images.acmesports.sports/Team+Golf+New+...,Team Golf New England Patriots Putter Grip,24.990000,0,1/6/2015 1:03,Standard Class


## Fix 1 : Column names to Snake_Case 

In [3]:
raw_data.columns.to_list()

['Type',
 'Days for shipping (real)',
 'Days for shipment (scheduled)',
 'Benefit per order',
 'Sales per customer',
 'Delivery Status',
 'Late_delivery_risk',
 'Category Id',
 'Category Name',
 'Customer City',
 'Customer Country',
 'Customer Email',
 'Customer Fname',
 'Customer Id',
 'Customer Lname',
 'Customer Password',
 'Customer Segment',
 'Customer State',
 'Customer Street',
 'Customer Zipcode',
 'Department Id',
 'Department Name',
 'Latitude',
 'Longitude',
 'Market',
 'Order City',
 'Order Country',
 'Order Customer Id',
 'order date (DateOrders)',
 'Order Id',
 'Order Item Cardprod Id',
 'Order Item Discount',
 'Order Item Discount Rate',
 'Order Item Id',
 'Order Item Product Price',
 'Order Item Profit Ratio',
 'Order Item Quantity',
 'Sales',
 'Order Item Total',
 'Order Profit Per Order',
 'Order Region',
 'Order State',
 'Order Status',
 'Order Zipcode',
 'Product Card Id',
 'Product Category Id',
 'Product Description',
 'Product Image',
 'Product Name',
 'Product P

In [4]:
raw_data.columns = (
    raw_data.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
)

In [5]:
raw_data.columns.to_list()

['type',
 'days_for_shipping_(real)',
 'days_for_shipment_(scheduled)',
 'benefit_per_order',
 'sales_per_customer',
 'delivery_status',
 'late_delivery_risk',
 'category_id',
 'category_name',
 'customer_city',
 'customer_country',
 'customer_email',
 'customer_fname',
 'customer_id',
 'customer_lname',
 'customer_password',
 'customer_segment',
 'customer_state',
 'customer_street',
 'customer_zipcode',
 'department_id',
 'department_name',
 'latitude',
 'longitude',
 'market',
 'order_city',
 'order_country',
 'order_customer_id',
 'order_date_(dateorders)',
 'order_id',
 'order_item_cardprod_id',
 'order_item_discount',
 'order_item_discount_rate',
 'order_item_id',
 'order_item_product_price',
 'order_item_profit_ratio',
 'order_item_quantity',
 'sales',
 'order_item_total',
 'order_profit_per_order',
 'order_region',
 'order_state',
 'order_status',
 'order_zipcode',
 'product_card_id',
 'product_category_id',
 'product_description',
 'product_image',
 'product_name',
 'product_p

## FIX 2 : Data Modeling

In [7]:

# ============================================================
# 0. SOURCE
# ============================================================

df = raw_data.copy()

# Clean column names
df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(" ", "_")
)

# ============================================================
# 1. TRASH / EXCLUDED COLUMNS
# ============================================================

trash_columns = [
    "customer_password",
    "order_zipcode",
    "product_description",
    "unnamed:_0",
    "sales_per_customer",
    "product_image"
]

# Only drop columns that actually exist
trash_columns = [c for c in trash_columns if c in df.columns]

dim_trash = df[trash_columns].copy()

# Remove trash from analytical source
df = df.drop(columns=trash_columns)


# ============================================================
# 2. DATE DIMENSION
# ============================================================

# Convert source dates
df["order_date_(dateorders)"] = pd.to_datetime(
    df["order_date_(dateorders)"],
    errors="coerce"
)

df["shipping_date_(dateorders)"] = pd.to_datetime(
    df["shipping_date_(dateorders)"],
    errors="coerce"
)

# Collect every date used by the model
min_date = min(
    df["order_date_(dateorders)"].min(),
    df["shipping_date_(dateorders)"].min()
)

max_date = max(
    df["order_date_(dateorders)"].max(),
    df["shipping_date_(dateorders)"].max()
)

date_range = pd.date_range(
    start=min_date.normalize(),
    end=max_date.normalize(),
    freq="D"
)

dim_date = pd.DataFrame({
    "full_date": date_range
})

dim_date["date_key"] = (
    dim_date["full_date"].dt.strftime("%Y%m%d").astype(int)
)

dim_date["year"] = dim_date["full_date"].dt.year
dim_date["quarter"] = "Q" + dim_date["full_date"].dt.quarter.astype(str)
dim_date["month"] = dim_date["full_date"].dt.month
dim_date["month_number"] = dim_date["full_date"].dt.month
dim_date["month_name"] = dim_date["full_date"].dt.month_name()
dim_date["week"] = dim_date["full_date"].dt.isocalendar().week.astype(int)
dim_date["day"] = dim_date["full_date"].dt.day
dim_date["day_name"] = dim_date["full_date"].dt.day_name()
dim_date["day_of_week"] = dim_date["full_date"].dt.dayofweek + 1
dim_date["is_weekend"] = (
    dim_date["full_date"].dt.dayofweek >= 5
)

# ============================================================
# 3. CUSTOMER DIMENSION
# ============================================================

customer_columns = [
    "customer_id",
    "customer_fname",
    "customer_lname",
    "customer_email",
    "customer_city",
    "customer_country",
    "customer_state",
    "customer_street",
    "customer_zipcode",
    "customer_segment"
]

customer_columns = [
    c for c in customer_columns
    if c in df.columns
]

dim_customer = (
    df[customer_columns]
    .drop_duplicates(subset=["customer_id"])
    .reset_index(drop=True)
)

# Surrogate key
dim_customer.insert(
    0,
    "customer_key",
    range(1, len(dim_customer) + 1)
)


# ============================================================
# 4. PRODUCT DIMENSION
# ============================================================

product_columns = [
    "product_card_id",
    "product_name",
    "product_price",
    "product_status",
    "product_category_id"
]

product_columns = [
    c for c in product_columns
    if c in df.columns
]

dim_product = (
    df[product_columns]
    .drop_duplicates(subset=["product_card_id"])
    .reset_index(drop=True)
)

dim_product.insert(
    0,
    "product_key",
    range(1, len(dim_product) + 1)
)


# ============================================================
# 5. CATEGORY DIMENSION
# ============================================================

category_columns = [
    "category_id",
    "category_name"
]

category_columns = [
    c for c in category_columns
    if c in df.columns
]

dim_category = (
    df[category_columns]
    .drop_duplicates(subset=["category_id"])
    .reset_index(drop=True)
)

dim_category.insert(
    0,
    "category_key",
    range(1, len(dim_category) + 1)
)


# ============================================================
# 6. DEPARTMENT DIMENSION
# ============================================================

department_columns = [
    "department_id",
    "department_name",
    "latitude",
    "longitude"
]

department_columns = [
    c for c in department_columns
    if c in df.columns
]

dim_department = (
    df[department_columns]
    .drop_duplicates(subset=["department_id"])
    .reset_index(drop=True)
)

dim_department.insert(
    0,
    "department_key",
    range(1, len(dim_department) + 1)
)


# ============================================================
# 7. DESTINATION DIMENSION
# ============================================================

destination_columns = [
    "market",
    "order_country",
    "order_state",
    "order_city",
    "order_region"
]

destination_columns = [
    c for c in destination_columns
    if c in df.columns
]

dim_destination = (
    df[destination_columns]
    .drop_duplicates()
    .reset_index(drop=True)
)

dim_destination.insert(
    0,
    "destination_key",
    range(1, len(dim_destination) + 1)
)


# ============================================================
# 8. SHIPPING DIMENSION
# ============================================================

dim_shipping = (
    df[["shipping_mode"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

dim_shipping.insert(
    0,
    "shipping_key",
    range(1, len(dim_shipping) + 1)
)


# ============================================================
# 9. ORDER STATUS DIMENSION
# ============================================================

dim_order_status = (
    df[["order_status"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

dim_order_status.insert(
    0,
    "order_status_key",
    range(1, len(dim_order_status) + 1)
)


# ============================================================
# 10. DELIVERY STATUS DIMENSION
# ============================================================

dim_delivery_status = (
    df[["delivery_status"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

dim_delivery_status.insert(
    0,
    "delivery_status_key",
    range(1, len(dim_delivery_status) + 1)
)


# ============================================================
# 11. CREATE LOOKUP DICTIONARIES
# ============================================================

customer_lookup = dict(
    zip(
        dim_customer["customer_id"],
        dim_customer["customer_key"]
    )
)

product_lookup = dict(
    zip(
        dim_product["product_card_id"],
        dim_product["product_key"]
    )
)

category_lookup = dict(
    zip(
        dim_category["category_id"],
        dim_category["category_key"]
    )
)

department_lookup = dict(
    zip(
        dim_department["department_id"],
        dim_department["department_key"]
    )
)

destination_lookup = dict(
    zip(
        dim_destination[
            destination_columns
        ].astype(str).agg("|".join, axis=1),
        dim_destination["destination_key"]
    )
)

shipping_lookup = dict(
    zip(
        dim_shipping["shipping_mode"],
        dim_shipping["shipping_key"]
    )
)

order_status_lookup = dict(
    zip(
        dim_order_status["order_status"],
        dim_order_status["order_status_key"]
    )
)

delivery_status_lookup = dict(
    zip(
        dim_delivery_status["delivery_status"],
        dim_delivery_status["delivery_status_key"]
    )
)

date_lookup = dict(
    zip(
        dim_date["full_date"],
        dim_date["date_key"]
    )
)


# ============================================================
# 12. PREPARE FOREIGN KEYS
# ============================================================

df["customer_key"] = df["order_customer_id"].map(
    customer_lookup
)

df["product_key"] = df["order_item_cardprod_id"].map(
    product_lookup
)

df["category_key"] = df["product_category_id"].map(
    category_lookup
)

df["department_key"] = df["department_id"].map(
    department_lookup
)

destination_key = (
    df[destination_columns]
    .astype(str)
    .agg("|".join, axis=1)
)

df["destination_key"] = destination_key.map(
    destination_lookup
)

df["shipping_key"] = df["shipping_mode"].map(
    shipping_lookup
)

df["order_status_key"] = df["order_status"].map(
    order_status_lookup
)

df["delivery_status_key"] = df["delivery_status"].map(
    delivery_status_lookup
)

df["order_date_key"] = (
    df["order_date_(dateorders)"]
    .dt.normalize()
    .map(date_lookup)
)

df["shipping_date_key"] = (
    df["shipping_date_(dateorders)"]
    .dt.normalize()
    .map(date_lookup)
)


# ============================================================
# 13. FACT ORDERS
# ============================================================
# GRAIN:
# One row = one order
# ============================================================

fact_orders_columns = [
    "order_id",
    "customer_key",
    "order_date_key",
    "destination_key",
    "shipping_key",
    "order_status_key",
    "delivery_status_key"
]

fact_orders = (
    df[fact_orders_columns]
    .drop_duplicates(subset=["order_id"])
    .reset_index(drop=True)
)


# ============================================================
# 14. FACT ORDER ITEMS
# ============================================================
# GRAIN:
# One row = one order item
# ============================================================

fact_order_items_columns = [
    "order_item_id",
    "order_id",

    "customer_key",
    "product_key",
    "category_key",
    "department_key",

    "order_date_key",
    "shipping_date_key",

    "order_item_discount",
    "order_item_discount_rate",
    "order_item_product_price",
    "order_item_profit_ratio",
    "order_item_quantity",
    "order_item_total",

    "sales",
    "order_profit_per_order",
    "benefit_per_order",

    "days_for_shipping_(real)",
    "days_for_shipment_(scheduled)",
    "late_delivery_risk",

    "type"
]

fact_order_items_columns = [
    c for c in fact_order_items_columns
    if c in df.columns
]

fact_order_items = (
    df[fact_order_items_columns]
    .drop_duplicates(subset=["order_item_id"])
    .reset_index(drop=True)
)


# ============================================================
# 15. CLEAN COLUMN TYPES
# ============================================================

# IDs → nullable integer
for table in [
    fact_orders,
    fact_order_items,
    dim_customer,
    dim_product,
    dim_category,
    dim_department,
    dim_destination,
    dim_shipping,
    dim_order_status,
    dim_delivery_status,
    dim_date
]:
    for col in table.columns:
        if col.endswith("_key"):
            table[col] = table[col].astype("Int64")


# ============================================================
# 16. DATA QUALITY CHECKS
# ============================================================

print("=" * 60)
print("STAR SCHEMA CREATED")
print("=" * 60)

print("\nDIMENSIONS")
print("-" * 60)

for name, table in {
    "dim_date": dim_date,
    "dim_customer": dim_customer,
    "dim_product": dim_product,
    "dim_category": dim_category,
    "dim_department": dim_department,
    "dim_destination": dim_destination,
    "dim_shipping": dim_shipping,
    "dim_order_status": dim_order_status,
    "dim_delivery_status": dim_delivery_status,
    "dim_trash": dim_trash
}.items():
    print(f"{name:<25} {table.shape}")

print("\nFACTS")
print("-" * 60)

print(
    f"{'fact_orders':<25} {fact_orders.shape}"
)

print(
    f"{'fact_order_items':<25} {fact_order_items.shape}"
)

print("\nGRAIN CHECKS")
print("-" * 60)

print(
    "Orders:",
    fact_orders["order_id"].nunique(),
    "unique orders /",
    len(fact_orders),
    "rows"
)

print(
    "Order items:",
    fact_order_items["order_item_id"].nunique(),
    "unique items /",
    len(fact_order_items),
    "rows"
)

print("\nFOREIGN KEY NULL CHECK")
print("-" * 60)

fk_columns = [
    "customer_key",
    "product_key",
    "category_key",
    "department_key",
    "destination_key",
    "shipping_key",
    "order_status_key",
    "delivery_status_key",
    "order_date_key",
    "shipping_date_key"
]

for col in fk_columns:
    if col in fact_order_items.columns:
        missing = fact_order_items[col].isna().sum()
        print(f"{col:<25} {missing} missing")

STAR SCHEMA CREATED

DIMENSIONS
------------------------------------------------------------
dim_date                  (1133, 12)
dim_customer              (20652, 11)
dim_product               (118, 6)
dim_category              (51, 3)
dim_department            (11, 5)
dim_destination           (3772, 6)
dim_shipping              (4, 2)
dim_order_status          (9, 2)
dim_delivery_status       (4, 2)
dim_trash                 (180519, 5)

FACTS
------------------------------------------------------------
fact_orders               (65752, 7)
fact_order_items          (180519, 21)

GRAIN CHECKS
------------------------------------------------------------
Orders: 65752 unique orders / 65752 rows
Order items: 180519 unique items / 180519 rows

FOREIGN KEY NULL CHECK
------------------------------------------------------------
customer_key              0 missing
product_key               0 missing
category_key              0 missing
department_key            0 missing
order_date_key      

## FIX 3 : Clean Data Exporting

In [8]:
# ============================================================
# EXPORT STAR SCHEMA TABLES TO CLEAN UTF-8 CSV FILES
# ============================================================

from pathlib import Path

# Output directory
output_path = Path("clean_data")

# Create directory if it doesn't exist
output_path.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# Tables to export
# ------------------------------------------------------------

tables = {
    "fact_orders": fact_orders,
    "fact_order_items": fact_order_items,

    "dim_date": dim_date,
    "dim_customer": dim_customer,
    "dim_product": dim_product,
    "dim_category": dim_category,
    "dim_department": dim_department,
    "dim_destination": dim_destination,
    "dim_shipping": dim_shipping,
    "dim_order_status": dim_order_status,
    "dim_delivery_status": dim_delivery_status,

    "dim_trash": dim_trash
}


# ------------------------------------------------------------
# Export
# ------------------------------------------------------------

for table_name, table in tables.items():

    file_path = output_path / f"{table_name}.csv"

    table.to_csv(
        file_path,
        index=False,
        encoding="utf-8-sig"
    )

    print(
        f"✓ {table_name:<25} "
        f"{len(table):>8,} rows  →  {file_path}"
    )


print("\n" + "=" * 65)
print("ALL STAR-SCHEMA TABLES EXPORTED SUCCESSFULLY")
print("=" * 65)

✓ fact_orders                 65,752 rows  →  clean_data\fact_orders.csv
✓ fact_order_items           180,519 rows  →  clean_data\fact_order_items.csv
✓ dim_date                     1,133 rows  →  clean_data\dim_date.csv
✓ dim_customer                20,652 rows  →  clean_data\dim_customer.csv
✓ dim_product                    118 rows  →  clean_data\dim_product.csv
✓ dim_category                    51 rows  →  clean_data\dim_category.csv
✓ dim_department                  11 rows  →  clean_data\dim_department.csv
✓ dim_destination              3,772 rows  →  clean_data\dim_destination.csv
✓ dim_shipping                     4 rows  →  clean_data\dim_shipping.csv
✓ dim_order_status                 9 rows  →  clean_data\dim_order_status.csv
✓ dim_delivery_status              4 rows  →  clean_data\dim_delivery_status.csv
✓ dim_trash                  180,519 rows  →  clean_data\dim_trash.csv

ALL STAR-SCHEMA TABLES EXPORTED SUCCESSFULLY
